In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/sample.csv.csv
/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/train.csv
/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/test.csv


In [ ]:
#Import basic libraries
import matplotlib.pyplot as plt
import seaborn as sns

According to the problem statement, Data compromises of two file:
- test.csv
- train.csv

Let's Load the data with the help of pandas dataframe

In [ ]:
#Read the datasets available for the problem statement
train_data=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/train.csv')
test_data=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/test.csv')

In order to understand the data, we will explore the `train_data` to understand the given data properly.

Let's start with the **Exploratory Data Analysis** 

## Exploratory Data Analysis

Let's start with understanding the dimensions  of the data

In [ ]:
#Check shape of the dataset
print(f'Shape of train dataset: \nNumber of rows: {train_data.shape[0]}\nNumber of Columns: {train_data.shape[1]}')

In `train_data`, We have $1,75,000$ rows and $17$ columns.

Let's list down all the columns available

In [ ]:
#Checking all the columns of the dataset
for i in train_data.columns:
    print(i)

According to the problem statement, target variable for this prediction task is `total_amount`

Let's have a 1st look of the `train_data`

In [ ]:
#Check 1st few entries of the dataset
train_data.head()

Let's understand the nature of numerical attributes of the `train_data`

In [ ]:
#Check numerical attribute of data
train_data.describe()

We will now explore every column present in the dataset to understand the distribution of data in each column

Let's analyze **`VendorID`** column

In [ ]:
#Let's look at VendorID column and it's data
print('Column: VendorID')
print(f'Datatype of columns: {train_data["VendorID"].dtype}')
print(f'Total number of unique values: {len(train_data["VendorID"].unique())}')
print(f'Unique Values are: {train_data["VendorID"].unique()}')

`VendorID` column has only 3 unique values i.e. `0`, `1` & `2` .

Let's look for any missing values in this column

In [ ]:
#Check for null values
train_data['VendorID'].isnull().sum()

There are `zero` null values for this column

Let's look at value counts for each unique values as well

In [ ]:
#Check for total value count for each vendors
value_count_vendors = train_data['VendorID'].value_counts()
value_count_vendors

Let's understand the portion of data for each `vendorID`

In [ ]:
#Plot pie chart to see distribution of vendors
plt.figure(figsize = (7,7))
plt.pie(value_count_vendors.values,autopct="%1.2f%%" ,labels=value_count_vendors.index)
plt.title('Vendors Distribution')

#Display the pie chart
plt.show()


Since data associated with `vendorId` is neligible, we will understand the distribution of our target variable `total_amount` for each `vendorId` to understand if there are underlying hidden pattern for any specific `vendorID`.

In [ ]:
#Plotting subplots as there are 3 vendors
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

#Plotting histograms for each VendorID
for vendor_id, ax in zip(range(2), axes):
    ax.hist(train_data[train_data['VendorID'] == vendor_id]['total_amount'],bins = 100 ,alpha=0.7,density = True)
    ax.set_title(f'Vendor {vendor_id}')
    ax.set_xlabel('Total Amount')
    ax.set_ylabel('Probability')

#Adjusting layout
plt.tight_layout()

#Display the subplots
plt.show()

In order to understand the above graph, Let's look at Mean and Variance.

In [ ]:
#Grouping the vendors by their Vendor ID
group_by_vendor = train_data.groupby(by='VendorID')

In [ ]:
#Check mean for each Vendor ID
mean_total_amount = group_by_vendor['total_amount'].mean()
mean_total_amount

In [ ]:
#Check variance for each Vendor ID
variance_total_amount = group_by_vendor['total_amount'].std()
variance_total_amount

We see that `total_amount` has different mean and variance so `vendorId` may be an important feature for our model training. However we will use feature selection techniques to decide on final features.

Now, Let's explore another column **`tpep_pickup_datetime`** and **`tpep_dropff_datetime`** 

In [ ]:
#Now, let's look at datetime column and understand it's data
print('Columns: tpep_pickup_datetime & tpep_dropoff_datetime')
print(f'Datatype of columns :\ntpep_pickup_datetime: {train_data["tpep_pickup_datetime"].dtype},\ntpep_dropoff_datetime: {train_data["tpep_dropoff_datetime"].dtype}')
print(f'Total no. of Pickup Datetime Unique Values: {len(train_data["tpep_pickup_datetime"].unique())}')
print(f'Total no. of Dropoff Datetime Unique Values: {len(train_data["tpep_dropoff_datetime"].unique())}')

Let's look for any missing values in this column

In [ ]:
#Check for null values
train_data[['tpep_pickup_datetime','tpep_dropoff_datetime']].isnull().sum()

There are `zero` missing values for above both tables

Since the dtype of above columns is object, Let's convert dtype of these columns to datetime

In [ ]:
#Let's change the datatype of column
train_data['tpep_pickup_datetime'] = pd.to_datetime(train_data['tpep_pickup_datetime'])
train_data['tpep_dropoff_datetime'] = pd.to_datetime(train_data['tpep_dropoff_datetime'])
train_data['tpep_pickup_datetime'].dtype, train_data['tpep_dropoff_datetime'].dtype

In [ ]:
#Creating a new column for duration
train_data['duration'] = (train_data['tpep_dropoff_datetime'] - train_data['tpep_pickup_datetime']).dt.total_seconds() / 60
train_data['duration'].head()

The result of the above data shows that for some of the columns with negative duration. This indicates that there are some anomalies while recording the pickup and dropoff times.

Since, we can't reach to the source to clearify this doubt, We will assume that the pick up date and drop off dates might have been swapped

Let's first check how many such wrong entries are there:

In [ ]:
#Checking anomalies in duration
(train_data['duration']<0).sum()

There are `65674` such entries are there where `tpep_dropofF_time` is earlier than `tpep_pickup_time`. Since Count of these entries are very large, In our analysis we will take the absolute difference of time to calculate the duration, considering that there might be some while recording the data.

We can't do much about this data as we don't have channels to communicate with the source of data on possible reason of discrepency for this dataset.

In [ ]:
#Rectifying data with our analysis
train_data['duration'] = abs(train_data['tpep_dropoff_datetime'] - train_data['tpep_pickup_datetime']).dt.total_seconds() / 60
train_data['duration'].head()

Now, Let' understand the distribution of the duration by ploting density plot

In [ ]:
#Plotting histogram to check probability density function of Duration
plt.figure(figsize = (7,7))
plt.hist(train_data['duration'],bins = 100 ,alpha=0.7,density = True,range = (0,500))
plt.title('Probability Density Function')
plt.xlabel('Duration (in minutes)')
plt.ylabel('Probability')

#Display the histogram
plt.show()

Let's look at presence of any outliers in the `duration`.

In [ ]:
#For checking outliers we plot Boxplot
plt.boxplot(train_data['duration'])

#Display the box plot
plt.show()

We see that we have many outliers for this column (`duration`)

Let's also look at the corelation between `duration` and `total_amount`; our target variable

In [ ]:
#Plotting scatter plot for total amount and duration
plt.figure(figsize=(8, 6))
plt.scatter(train_data['duration'], train_data['total_amount'], alpha=0.8)
plt.xlabel('duration (in minutes)')
plt.ylabel('total_amount')

#Display the scatter plot
plt.show()

Let's extract few more potential features from these columns such as date of booking and time of booking

In [ ]:
#Creating new column for pickup date and time
train_data['pickup_date'] = train_data['tpep_pickup_datetime'].dt.date
train_data['pickup_hour'] = train_data['tpep_pickup_datetime'].dt.hour

Let's look at value count of total booking per day 


In [ ]:
#Check for total value count for each pickup date
train_data['pickup_date'].value_counts()

Here we see that dataset available has booking from just 4 days with highest booking on `29 June 2023` and `30 June 2023`.

Also, In order to understand peak booking hours, let's plot frequency plot of `pickup_hour`.

In [ ]:
#Plotting histogram to check peak booking hours
plt.hist(train_data['pickup_hour'], bins = 24, alpha = 0.7)
plt.title('Frequency Plot')
plt.xlabel('Time')
plt.ylabel('Frequency')

#Adjusting histogram
plt.tight_layout()

#Display histogram
plt.show()

We can classify the booking time into three bins i.e `low`, `moderate` and `high` peak timings and will use this feature for our model training. We understand that this is not the ideal method to model the peak timings in real scenerio however history can be starting point to model peak and at latter stage real time information can be used if one has the access to that data. 

**`low` peak from the data  can fall in the range timing of $[0,8]$, booking between $[8-15]$ can be classifies as `moderate` peak and booking between $[15-24]$ can be considered `high` peak.**

Let's explore another column **`passenger_count`**

In [ ]:
#Now let's look at passenger_count column and understand it's data
print('Column : passenger_count')
print(f'Datatype of columns : {train_data["passenger_count"].dtype}')
print(f'Total number of unique values : {len(train_data["passenger_count"].unique())}')
print(f'Unique Values are : {train_data["passenger_count"].unique()}')

From the unique values, we see that there are a few `nan` values. Let's see how many values are missing

In [ ]:
#Check for null values
train_data['passenger_count'].isnull().sum()

There are total $6077$ missing values for passenger count. We need to `impute` the nan values while model training

Let's look at value counts for each unique values as well

In [ ]:
#Check for total value count for each passenger
value_count_passengers = train_data['passenger_count'].value_counts()
value_count_passengers

In [ ]:
#Plot pie chart to see distribution of passenger count
plt.figure(figsize=(7,7))
plt.pie(value_count_passengers.values ,labels=value_count_passengers.index, pctdistance=0.85)
plt.title('Passengers Distribution')

#Display pie chart
plt.show()


Above pie plot shows portion ratio passenger count through out the dataset 

Let's understand the distribution of our target variable for different value of `passenger_count`

In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15,15))
axes = axes.flatten()

values = train_data["passenger_count"].unique()
# values.dtype = float

# Plotting histograms for each passenger
for passenger_count, ax in zip(values, axes):
    if np.isnan(passenger_count):
        ax.hist(train_data[train_data['passenger_count'].isna()]['total_amount'],bins = 100 ,alpha=0.7)
    else:
        ax.hist(train_data[train_data['passenger_count'] == passenger_count]['total_amount'],bins = 100 ,alpha=0.7)

    ax.set_title(f'Passenger Count {passenger_count}')
    ax.set_xlabel('Total Amount')
    ax.set_ylabel('Frequency')

# Adjusting layout
plt.tight_layout()

# Display the subplots
plt.show()

Let's explore another column **`trip_distance`**

In [ ]:
#Now, let's look at trip_distance column and understand it's data
print('Column: trip_distance')
print(f'Datatype of columns: {train_data["trip_distance"].dtype}')
print(f'Total number of unique values: {len(train_data["trip_distance"].unique())}')

Let's look for missing values in this column

In [ ]:
#Check for null values
train_data['trip_distance'].isnull().sum()

There are `zero` missing values

Now Let's looks for potential outliers for `trip distance`

In [ ]:
#Plotting box plot
plt.boxplot(train_data['trip_distance'])
plt.title('Box Plot of Trip Distance')

#Display box plot
plt.show()

Outliers are present in the column `trip_distance` as visible by the box plot above. We will handle these outliers while data cleaning.

Also, Let's look for its distribution

In [ ]:
#Plotting Histogram to look at it's distribution
plt.hist(train_data['trip_distance'], bins = 100, range=(0,50))
plt.title('Histogram of Trip Distance')
plt.xlabel('Trip Distance')
plt.ylabel('Frequency')

#Display Histogram
plt.show()

Let's look at another column **`RateCodeID`**

In [ ]:
#Now let's look at RatecodeID column and understand it's data
print('Column: RatecodeID')
print(f'Datatype of columns: {train_data["RatecodeID"].dtype}')
print(f'Total number of unique values: {len(train_data["RatecodeID"].unique())}')
print(f'Unique Values are: {train_data["RatecodeID"].unique()}')

We see that there is an outlier i.e value `99`. Since generally rating happens to be between 1-5. We will take care of this value. 

Also we see that there is presence of `nan` value here. Let's see total count of `nan` values 

In [ ]:
#check for null values
train_data["RatecodeID"].isnull().sum()

There are about `6077` rows for which values are missing

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each rating
value_count_rating = train_data['RatecodeID'].value_counts()
value_count_rating

We will replace the any value bigger than 5 with mean `RateCodeID` during data cleaning phase

Let's look for another column **`store_and_fwd_flag`**

In [ ]:
#Now ,let's look at store_and_fwd_flag column and understand it's data
print('Column: store_and_fwd_flag')
print(f'Datatype of columns: {train_data["store_and_fwd_flag"].dtype}')
print(f'Total number of unique values: {len(train_data["store_and_fwd_flag"].unique())}')
print(f'Unique Values are: {train_data["store_and_fwd_flag"].unique()}')

From the unique values, we see that there are a few nan values. Let's see how many values are missing

In [ ]:
#check for null values
train_data["store_and_fwd_flag"].isnull().sum()

There are total `6077` missing values for store and fwd flag. We need to impute the nan values while model training

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each flag
value_count_flag = train_data['store_and_fwd_flag'].value_counts()
value_count_flag

Now, let's analyze **`PULocationID`** and **`DULocationID`** columns

In [ ]:
#Now, let's look at LocationID columns and understand it's data
print('Columns: PULocationID & DOLocationID')
print(f'Datatype of columns: {train_data["PULocationID"].dtype}, {train_data["DOLocationID"].dtype}')
print(f'Total number of unique values: {len(train_data["PULocationID"].unique())}, {len(train_data["DOLocationID"].unique())}')

Let's look for any missing values in these columns

In [ ]:
print(f'Total number of missing values: {(train_data["PULocationID"].isnull().sum())}, {(train_data["DOLocationID"].isnull().sum())}')

There are `zero` null values in this column

Now, let's analyze another column **`payment_type`**

In [ ]:
#Now let's look at payment_type column and understand it's data
print('Column: payment_type')
print(f'Datatype of columns: {train_data["payment_type"].dtype}')
print(f'Total number of unique values: {len(train_data["payment_type"].unique())}')
print(f'Unique Values are: {train_data["payment_type"].unique()}')

Let's look for any missing values in these column

In [ ]:
train_data['payment_type'].isnull().sum()

There are `zero` null values in this column

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each payment type
value_count_payment_type = train_data['payment_type'].value_counts()
value_count_payment_type

In [ ]:
#Plot pie chart to see distribution of payment type
plt.figure(figsize = (7,7))
plt.pie(value_count_payment_type.values, labels = value_count_payment_type.index, autopct = "%1.2f%%")

#Display pie chart
plt.show()

Above pie plot shows portion ratio payment type through out the dataset

Let's understand the distribution of our target variable for different value of `payment_type`

In [ ]:
#Plotting subplots as there are 5 payment types
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15,15))
axes = axes.flatten()

values = train_data["payment_type"].unique()
#values.dtype = float

#Plotting histograms for each payment type
for payment_type, ax in zip(values, axes):
    ax.hist(train_data[train_data['payment_type'] == payment_type]['total_amount'],bins = 100 ,alpha=0.7)
    ax.set_title(f'Payment Type: {payment_type}')
    ax.set_xlabel('Total Amount')
    ax.set_ylabel('Frequency')

#Adjusting layout
plt.tight_layout()

#Display the subplots
plt.show()

Now, Let's analyze another column **`extra`**

In [ ]:
#Now, let's look at extra column and understand it's data
print('Column: extra')
print(f'Datatype of columns: {train_data["extra"].dtype}')
print(f'Total number of unique values: {len(train_data["extra"].unique())}')

Let's look for any missing values in this column

In [ ]:
#Check for null values
train_data['extra'].isnull().sum()

There are `zero` null values in this column

Now, Let' understand the distribution of the duration by ploting density plot

In [ ]:
#Plotting Histogram to check probability density function of extra
plt.hist(train_data['extra'], density = True)
plt.title('Probability Density Function')
plt.xlabel('Extra')
plt.ylabel('Probability')


#Display histogram
plt.show()

Now Let's looks for potential outliers for `extra`

In [ ]:
#Plotting box plot
plt.boxplot(train_data['extra'])

#Display Box Plot
plt.show()

We see that we have many outliers for this column (extra)

Now Let's analyze **`tip_amount`** column

In [ ]:
#Now let's look at tip_amount column and understand it's data
print('Column: tip_amount')
print(f'Datatype of columns: {train_data["tip_amount"].dtype}')
print(f'Total number of unique values: {len(train_data["tip_amount"].unique())}')

Let's look for any missing values in this column

In [ ]:
#Check for null values
train_data['tip_amount'].isnull().sum()

There are `zero` null values in this column

In order to understand tip amount, let's plot frequency plot of `tip_amount`.

In [ ]:
#Plotting histogram to check highest tip amount
plt.hist(train_data['tip_amount'], range(0, 50))
plt.title('Frequency Plot')
plt.xlabel('Tip Amount')
plt.ylabel('Frequency')

#Display Histogram
plt.show()

 Let's look at presence of any outliers in the `tip_amount`

In [ ]:
#Plotting Box plot
plt.boxplot(train_data['tip_amount'])

#Display box plot
plt.show()

We see that we have many outliers for this column (tip_amount)

Now, Let's analyze **`tolls_amount`** column

In [ ]:
#Now let's look at tolls_amount column and understand it's data
print('Column: tolls_amount')
print(f'Datatype of columns: {train_data["tolls_amount"].dtype}')
print(f'Total number of unique values: {len(train_data["tolls_amount"].unique())}')

Let's look for any missing values in this column

In [ ]:
#Check for null values
train_data['tolls_amount'].isnull().sum()

There are `zero` missing values for above both tables

 In order to understand tip amount, let's plot frequency plot of `tolls_amount`.

In [ ]:
#Plotting histogram to check highest tip amount
plt.hist(train_data['tolls_amount'], range=(-20,40))
plt.title('Frequency Plot')
plt.xlabel('Tolls Amount')
plt.ylabel('Frequency')

#Display histogram
plt.show()

Let's look at presence of any outliers in the `tolls_amount`

In [ ]:
#Plotting Box plot
plt.boxplot(train_data['tolls_amount'])

#Display box plot
plt.show()

We see that we have many outliers for this column (tolls_amount)

In [ ]:
#Now let's look at improvement_surcharge column and understand it's data
print('Column: improvement_surcharge')
print(f'Datatype of columns: {train_data["improvement_surcharge"].dtype}')
print(f'Total number of unique values: {len(train_data["improvement_surcharge"].unique())}')
print(f'Unique Values are: {train_data["improvement_surcharge"].unique()}')

Let's look for any missing values in this column

In [ ]:
train_data['improvement_surcharge'].isnull().sum()

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each improvement surcharge
value_count_improvement_surcharge = train_data['improvement_surcharge'].value_counts()
value_count_improvement_surcharge

Now, Let's explore another column **`congestion_surcharge`**

In [ ]:
#Now let's look at congestion_surcharge column and understand it's data
print('Column : congestion_surcharge')
print(f'Datatype of columns : {train_data["congestion_surcharge"].dtype}')
print(f'Total number of unique values : {len(train_data["congestion_surcharge"].unique())}')
print(f'Unique Values are : {train_data["congestion_surcharge"].unique()}')

From the unique values, we see that there are a few `nan` values. Let's see how many values are missing

In [ ]:
#check for null values
train_data['congestion_surcharge'].isnull().sum()

There are `6077` null values for this column

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each congestionsurcharge
value_count_congestion_surcharge = train_data['congestion_surcharge'].value_counts()
value_count_congestion_surcharge

Now, Let's explore another column **`Airport_fee`**

In [ ]:
#Now let's look at Airport_fee and understand it's data
print('Column : Airport_fee')
print(f'Datatype of columns : {train_data["Airport_fee"].dtype}')
print(f'Total number of unique values : {len(train_data["Airport_fee"].unique())}')
print(f'Unique Values are : {train_data["Airport_fee"].unique()}')

From the unique values, we see that there are a few `nan` values. Let's see how many values are missing

In [ ]:
#check for null values
train_data['Airport_fee'].isnull().sum()

There are `6077` null values for this column

Let's look at value counts for each unique values

In [ ]:
#Check for total value count for each Airport fee
value_count_airport_fee = train_data['Airport_fee'].value_counts()
value_count_airport_fee

As we have good understanding of data. Let's start with Data cleaning and Feature Extraction.

## Data Preprocessing

In [ ]:
train_dataset=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/train.csv')
test_dataset=pd.read_csv('/kaggle/input/taxi-fare-guru-total-amount-prediction-challenge/test.csv')

In [ ]:
def preprocess_data(dataset):
    dataset['tpep_pickup_datetime'] = pd.to_datetime(dataset['tpep_pickup_datetime'])
    dataset['tpep_dropoff_datetime'] = pd.to_datetime(dataset['tpep_dropoff_datetime'])
    dataset['duration'] = (dataset['tpep_dropoff_datetime'] - dataset['tpep_pickup_datetime']).dt.total_seconds() / 60
    dataset['pickup_hour'] = dataset['tpep_pickup_datetime'].dt.hour
    label = ['low', 'moderate', 'high']
    bins = [0, 8, 15, 24]
    dataset['peak_bookings'] = pd.cut(dataset['pickup_hour'], bins, labels = label, include_lowest=True, right=False)  
    
    return dataset

In [ ]:
train_dataset_preprocessed = preprocess_data(train_dataset)
test_processed_data = preprocess_data(test_dataset)

## Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model  import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from xgboost import XGBRegressor

In [ ]:
features = ['passenger_count', 'trip_distance', 'RatecodeID', 'payment_type', 'extra', 'tip_amount',
       'tolls_amount', 'improvement_surcharge',
       'congestion_surcharge', 'Airport_fee', 'duration',
       'peak_bookings']

label = 'total_amount'

In [ ]:
train_feature_dataset = train_dataset_preprocessed[features]
train_labels = train_dataset_preprocessed[label]

In [ ]:
train_feature_dataset.shape , train_labels.shape

In [ ]:
train_feature_dataset.head()

In [ ]:
categorical_features = ['payment_type','peak_bookings']
numerical_features = ['passenger_count','trip_distance','RatecodeID','extra',
                      'tip_amount','tolls_amount','improvement_surcharge','congestion_surcharge',
                     'Airport_fee','duration']

In [ ]:
numerical_pipeline = Pipeline([
                               ('imputer', SimpleImputer(strategy="mean")),
                               ('std_scaler', StandardScaler()),])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_pipeline = ColumnTransformer([
    ("num",numerical_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

preprocess_pipeline


## Model Building

In [ ]:
X_train, X_test,y_train,y_test = train_test_split(train_feature_dataset,train_labels,test_size = 0.3, random_state = 42)

In [ ]:
print(f'Shape of training set: {X_train.shape,y_train.shape}')
print(f'Shape of test set: {X_test.shape,y_test.shape}')

### Linear Regression

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline),
    ('linear_reg',LinearRegression())
])

In [ ]:
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
linear_reg_r2_score = r2_score(y_test,y_pred)
linear_reg_r2_score

#### Linear Regression with Polynomial Features

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline),
    ('poly_features', PolynomialFeatures()),
    ('std_sclar',StandardScaler()),
    ('linear_reg',LinearRegression())
])

In [ ]:
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
poly_reg_r2_score = r2_score(y_test,y_pred)
poly_reg_r2_score

In [ ]:
param_grid = {
    'poly_features__degree': [1,2, 3], 
}
# kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters: ", grid_search.best_params_)
print("Best Score: ", grid_search.best_score_)


In [ ]:
best_poly_reg_model_pipeline = grid_search.best_estimator_
y_pred = best_poly_reg_model_pipeline.predict(X_test)
best_poly_reg_r2_score = r2_score(y_test,y_pred)
best_poly_reg_r2_score

### SGDRegressor

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline), 
    ('sgd_reg', SGDRegressor(loss = 'huber'))
])
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
linear_reg_with_sgd_r2_score = r2_score(y_test,y_pred)
linear_reg_with_sgd_r2_score

In [ ]:
param_grid = {
    'sgd_reg__loss': ['squared_error', 'huber'],
    'sgd_reg__alpha':[0.0001,0.001,0.01,0.1,1],
    'sgd_reg__eta0': [0.001,0.01, 0.1],
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(model_pipeline, param_grid, cv=kf, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters: ", grid_search.best_params_)
print("Best Score: ", grid_search.best_score_)

In [ ]:
best_sgd_model_pipeline = grid_search.best_estimator_
best_sgd_model_pipeline.fit(X_train,y_train)
y_pred = best_sgd_model_pipeline.predict(X_test)
linear_reg_with_best_sgd_r2_score = r2_score(y_test,y_pred)
linear_reg_with_best_sgd_r2_score

### Decision Tree Regressor and Random Forest Regressor

#### Decision Tree Regressor

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline),
    ('DecisionTreeRegressor',DecisionTreeRegressor())
])

In [ ]:
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
r2_score(y_test,y_pred)

#### Random Forest Regressor

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline),
    ('RandomForest',RandomForestRegressor(max_features='sqrt'))
])

In [ ]:
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
r2_score(y_test,y_pred)

Since vanila Random Forest Regressor is giving better score than vanila decision tree we will train our hyperparameters of random forest regressor to get better result

In [ ]:
param_grid = {
    'RandomForest__n_estimators': [150, 100],          
    'RandomForest__max_depth': [20,10, None],
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(model_pipeline, param_grid, cv=kf, scoring='r2')
grid_search.fit(X_train, y_train)
print("Best Parameters: ", grid_search.best_params_)
print("Best Score: ", grid_search.best_score_)

In [ ]:
best_random_forest_model_pipeline = grid_search.best_estimator_
best_random_forest_model_pipeline.fit(X_train,y_train)
y_pred = best_random_forest_model_pipeline.predict(X_test)
random_forest_with_best_r2_score = r2_score(y_test,y_pred)
random_forest_with_best_r2_score

### XGBRegressor

In [ ]:
model_pipeline = Pipeline([
    ('preprocess',preprocess_pipeline),
    ('xgBoost',XGBRegressor())
])

In [ ]:
model_pipeline.fit(X_train,y_train)
y_pred = model_pipeline.predict(X_test)
r2_score(y_test,y_pred)

In [ ]:
param_grid = {
    'xgBoost__learning_rate': [0.01, 0.1, 0.2],
    'xgBoost__n_estimators': [50, 100, 200],
    'xgBoost__max_depth': [3, 4, 5],
    'xgBoost__subsample': [0.8, 0.9, 1.0],
    
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(model_pipeline, param_grid, cv=kf, scoring='r2')
grid_search.fit(X_train, y_train)
print("Best Parameters: ", grid_search.best_params_)
print("Best Score: ", grid_search.best_score_)

In [ ]:
best_xgboost_model_pipeline = grid_search.best_estimator_
best_xgboost_model_pipeline.fit(X_train,y_train)
y_pred = best_xgboost_model_pipeline.predict(X_test)
xgboost_with_best_r2_score = r2_score(y_test,y_pred)
xgboost_with_best_r2_score

Let's compare the r2 scores from different models

In [ ]:
Results = {'Linear Regression': linear_reg_r2_score,
          'Linear Regression with Polynomial Features': best_poly_reg_r2_score,
           'SGDRegressor': linear_reg_with_best_sgd_r2_score,
           'Random Forest Regressor': random_forest_with_best_r2_score,
           'XGBRegressor': xgboost_with_best_r2_score
          }

results_df = pd.DataFrame(list(Results.items()), columns=['Model', 'R2 Score'])
results_df

Since, the r2 score on the test data is highest with XGBRegressor  will train our final model with XGBRegressor.

In [ ]:
model_pipeline = best_xgboost_model_pipeline

In [ ]:
#Don't delete this code block
model_pipeline.fit(train_feature_dataset,train_labels)
y_pred = model_pipeline.predict(test_processed_data)

In [ ]:
#Don't delete this code
submission = pd.DataFrame({"ID": np.arange(1,test_processed_data.shape[0]+1), 
                           "total_amount": y_pred}) 

submission.to_csv('submission.csv',index=False)